In [ ]:
import pandas as pd
import numpy as np
import os
from matplotlib import pyplot as plt
import decoding_utils as du
%matplotlib inline 

In [ ]:
import vbn_utils

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/vbn_s3_cache/visual-behavior-neuropixels-0.4.0/project_metadata/ecephys_sessions.csv"

In [ ]:
units = pd.read_csv(unit_table_file)

In [ ]:
#Clean layers
units['cortical_layer'].replace('3-Feb', '2/3', inplace=True)
units['cortical_layer'].replace('6a', '6', inplace=True)
units['cortical_layer'].replace('6b', '6', inplace=True)

In [ ]:
stim_table = pd.read_csv(stim_table_file)

In [ ]:
sessions_table = pd.read_csv(sessions_table_file)

In [ ]:
g_images = ['omitted'] + list(np.sort(stim_table[(stim_table['stimulus_name'].str.contains('_G_'))&
                    (~stim_table['omitted'])&
                    (~stim_table['image_name'].isin(['im083_r','im111_r']))]['image_name'].unique())) + ['im083_r','im111_r']

h_images = ['omitted'] + list(np.sort(stim_table[(stim_table['stimulus_name'].str.contains('_H_'))&
                    (~stim_table['omitted'])&
                    (~stim_table['image_name'].isin(['im083_r','im111_r']))]['image_name'].unique())) + ['im083_r','im111_r']

### Generated on HPC by 'run_image_decoding.py'

In [ ]:
image_decoding_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/image_decoding"
decoding_files = os.listdir(image_decoding_dir)
decoding_files = [os.path.join(image_decoding_dir, f) for f in decoding_files if f.endswith('nonchangeRS.npy')]

session_ids = [int(os.path.basename(f).split('_')[0]) for f in decoding_files]

missing_sessions = [s for s in sessions_table['ecephys_session_id'].values if s not in session_ids]
missing_sessions

In [ ]:
decoding_results_files = os.listdir("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/image_decoding")
decoding_results_files = [os.path.join("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/image_decoding", d) for d in decoding_results_files]

In [ ]:
no_anomalies = sessions_table[(sessions_table['abnormal_activity'].isnull())&(sessions_table['abnormal_histology'].isnull())]

In [ ]:
from notebook_utils import (
    get_decoding_results_files, get_mouse_paired_indices
)

In [ ]:
conditions = ['Familiar', 'Novel']

In [ ]:
image_set = 'nonchangeRS' 
session_decoding_files = get_decoding_results_files(no_anomalies['ecephys_session_id'].values, decoding_results_files, image_set=image_set)
decoding_results_dict = {}
for decoding_file in session_decoding_files:

    d = np.load(decoding_file, allow_pickle=True).item()
    decoding_results_dict[decoding_file] = d

decode_windows = d['decodeWindows']

In [ ]:
gtoh_sessions = no_anomalies[((no_anomalies['image_set']=='G')&(no_anomalies['experience_level']=='Familiar')) | 
                            (((no_anomalies['image_set']=='H')&(no_anomalies['experience_level']=='Novel')))]

htog_sessions = no_anomalies[((no_anomalies['image_set']=='H')&(no_anomalies['experience_level']=='Familiar')) | 
                            (((no_anomalies['image_set']=='G')&(no_anomalies['experience_level']=='Novel')))]

In [ ]:
regions = list(decoding_results_dict[decoding_file].keys())
regions = [r for r in regions if r not in ['decodeWindows', 'hit', 'image_order']]
unit_sub_sample_sizes = list(decoding_results_dict[decoding_file][regions[0]].keys())

conditions = ['Familiar', 'Novel']

region_timecourses = {}
session_ids = {r:{c: {n:[] for n in unit_sub_sample_sizes} for c in conditions} for r in regions}
for region in regions:
    imagewise_recall_per_condition = {c:{} for c in conditions}
    for condition in conditions:

        condition_sessions = no_anomalies[no_anomalies['experience_level']==condition]
        condition_decoding_files = get_decoding_results_files(condition_sessions['ecephys_session_id'].values, decoding_results_files, image_set=image_set)
        
        for n in unit_sub_sample_sizes:
            bas = []
            sess_ids = []
            for cdf in condition_decoding_files:
                d = decoding_results_dict[cdf]
                ba = np.array(d[region][n]['imagewise_recall'])
                #ba = np.array(d[region][n]['imagewise_precision']) #can also plot precision
                if ba.size>0:
                    bas.append(ba)
                    sess_ids.append(int(os.path.basename(cdf).split('_')[0]))

            ba_array = np.array(bas)
            imagewise_recall_per_condition[condition][n] = ba_array
            session_ids[region][condition][n] = sess_ids
        
    region_timecourses[region] = imagewise_recall_per_condition

In [ ]:
from notebook_utils import (
    
    get_sigmoidfit_midpoint,
    
)

In [ ]:
for region in regions:
    fig, axes = plt.subplots(1,len(unit_sub_sample_sizes))
    fig.set_size_inches(18,6)
    imagewise_recall_per_condition = region_timecourses[region]

    for n, ax in zip(unit_sub_sample_sizes, axes):
        colors = ['b', 'r']
        counts = []
        for ic, condition in enumerate(['Familiar', 'Novel']):
            count = imagewise_recall_per_condition[condition][n].shape[0]
            counts.append(count)
            if count>0:
                mean = np.mean(imagewise_recall_per_condition[condition][n][:, :, :6], axis=(0,2))
                std = np.std(imagewise_recall_per_condition[condition][n][:, :, :6], axis=(0,2))
                sem = std/count**0.5
                time = np.arange(0, len(mean)*10, 10)
                ax.plot(time, mean, colors[ic])
                ax.fill_between(time, mean+sem, mean-sem, color=colors[ic], alpha=0.3)
        
        ax.set_title(f'{region} {n}, f: {counts[0]}, n: {counts[1]}')
        ax.set_xlim([0,150])
        ax.set_ylim([0,1])


In [ ]:
#accuracy at end of decision window (100ms) and n=40
hit_rates = {r: {c:{n:[] for n in unit_sub_sample_sizes} for c in ['Familiar', 'Novel']} for r in regions}
for region in regions:
    for n in [40,20]:
        for condition in conditions:
            timecourse = region_timecourses[region][condition][n]
            count = timecourse.shape[0]
            if count>0:
                timecourse_means = np.mean(timecourse[:,:,:6], axis=2)
                hrs = timecourse_means[:, 10]
                hit_rates[region][condition][n] = hrs
                

In [ ]:
from notebook_utils import bh_multitest
import scipy.stats

In [ ]:
regions_to_plot = ['LGd', 'LP', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'VISall', 'SCMRN']
fig, ax = plt.subplots()
pvals = []
for ir, region in enumerate(regions_to_plot):#regions_to_plot:

    vals = [np.array(hit_rates[region][cond][40]) for cond in ['Familiar', 'Novel']]
    pval = scipy.stats.ranksums(*vals, nan_policy='omit')[1]

    pvals.append(pval)
    
    ax.plot(ir, np.nanmean(vals[0]), 'bo')
    ax.plot(ir+0.1, np.nanmean(vals[1]), 'ro')

    ax.errorbar(ir, np.nanmean(vals[0]), np.nanstd(vals[0])/(np.sum(~np.isnan(vals[0]))**0.5), color='b')
    ax.errorbar(ir+0.1, np.nanmean(vals[1]), np.nanstd(vals[1])/(np.sum(~np.isnan(vals[1]))**0.5), color='r')

ax.set_xticks(np.arange(len(regions_to_plot)))
ax.set_xticklabels(regions_to_plot, rotation=90)
ax.set_ylabel('Decoding accuracy')

sig_after_correction = bh_multitest(pvals)[0]
sigx_ind = np.where(sig_after_correction)[0]
if len(sigx_ind)>0:
    ax.text(sigx_ind[0], 1, '*')

vbn_utils.formatFigure(fig, ax)

In [ ]:
colors = ['b', 'r']
plt.rcParams['font.size'] = 20
latencies = {r: {c:{n:[] for n in unit_sub_sample_sizes} for c in ['Familiar', 'Novel']} for r in regions}
latency_sessions = {r: {c:{n:[] for n in unit_sub_sample_sizes} for c in ['Familiar', 'Novel']} for r in regions}

for region in ['LGd', 'VISp', 'VISl', 'VISall']:
    imagewise_recall_per_condition = region_timecourses[region]
    plt.figure()
    fig = plt.gcf()
    fig.set_size_inches([8,6])
    counts=[]
    for n in [40]:
        for ic, condition in enumerate(['Familiar', 'Novel']):
            print(imagewise_recall_per_condition[condition][n].shape)
            count = imagewise_recall_per_condition[condition][n].shape[0]
            counts.append(count)
            if count>0:
                image_means = np.mean(imagewise_recall_per_condition[condition][n][:, :, :6], axis=(2))
                lats = [get_sigmoidfit_midpoint(decode_windows, y)[0] for y in image_means]
                ylats = [get_sigmoidfit_midpoint(decode_windows, y)[1] for y in image_means]
                plt.plot(decode_windows, image_means.T, colors[ic], alpha=0.2)
                plt.plot(lats, ylats, colors[ic]+'o', alpha=0.5, mec='none')
                
                latencies[region][condition][n] = lats
                plt.plot(decode_windows, np.mean(image_means, axis=0), colors[ic], lw=2)
                print(f'{region}: {np.median(lats)}')
    ax = plt.gca()
    ax.set_title(f'{region} {n} units, f: {counts[0]}, n: {counts[1]}')
    ax.set_xlim([0, 120])
    ax.set_xticks([20, 100])
    ax.spines['bottom'].set_bounds(20, 100)
    vbn_utils.formatFigure(fig, ax)
    ax.set_xlabel('Time from stim start (ms)')
    ax.set_ylabel('Decoding accuracy')

    # Set the linewidth of all spines
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)  # Set spine linewidth to 2 points

    # Set the linewidth of ticks
    ax.tick_params(width=1.5)

In [ ]:
latencies = {r: {c:{n:[] for n in unit_sub_sample_sizes} for c in ['Familiar', 'Novel']} for r in regions}
latency_sessions = {r: {c:{n:[] for n in unit_sub_sample_sizes} for c in ['Familiar', 'Novel']} for r in regions}

for region in regions:
    imagewise_recall_per_condition = region_timecourses[region]

    for n in [40,20]:#[10,20,40]:
        for ic, condition in enumerate(['Familiar', 'Novel']):
            count = imagewise_recall_per_condition[condition][n].shape[0]
            counts.append(count)
            if count>0:
                image_means = np.mean(imagewise_recall_per_condition[condition][n][:, :, :6], axis=(2))
                lats = [get_sigmoidfit_midpoint(decode_windows, y)[0] for y in image_means]
                
                latencies[region][condition][n] = lats


In [ ]:
regions_to_plot = ['LGd', 'LP', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'VISall', 'SCMRN']
region_means = []
plt.figure()
pvals = []
for ir, region in enumerate(regions_to_plot):#regions:

    vals = [np.array(latencies[region][cond][20]) for cond in ['Familiar', 'Novel']]
    pvals.append(scipy.stats.mannwhitneyu(*vals, nan_policy='omit')[1])
    # print(ir)
    # print(np.nanmean(vals[0]))
    plt.plot(ir, np.nanmean(vals[0]), 'bo')
    plt.plot(ir+0.1, np.nanmean(vals[1]), 'ro')

    plt.errorbar(ir, np.nanmean(vals[0]), np.nanstd(vals[0])/(np.sum(~np.isnan(vals[0]))**0.5), color='b')
    plt.errorbar(ir+0.1, np.nanmean(vals[1]), np.nanstd(vals[1])/(np.sum(~np.isnan(vals[1]))**0.5), color='r')

ax = plt.gca()
ax.set_xticks(np.arange(len(regions_to_plot)))
ax.set_xticklabels(regions_to_plot, rotation=90)
ax.set_ylabel('Decoding latency (ms)')

sig_after_correction = bh_multitest(pvals)[0]
sigx_ind = np.where(sig_after_correction)[0]
[ax.text(x, ax.get_ylim()[1], '*') for x in sigx_ind]
vbn_utils.formatFigure(fig, ax)